# Laboratorio 4 — Análisis de Datos GeoEspaciales
## Notebook 1: Conexión a la API y descarga de datos (Ejercicios 1 y 2)

Lagos Atitlán y Amatitlán — detección de floraciones de cianobacteria con Sentinel-2 (Copernicus Data Space Ecosystem).

**Ejercicio 1.** Establecer conexión con la API de Sentinel-2 usando `openeo`.

**Ejercicio 2.** Obtener únicamente los datos raster necesarios para cada lago (bandas B03, B04, B08 para NDVI/NDWI vía openEO; resultado del script oficial de cianobacteria vía Sentinel Hub Process API), usando las fechas oficiales provistas en el enunciado.

### 0. Setup

Requiere las variables de entorno `SH_CLIENT_ID` y `SH_CLIENT_SECRET` (ver `.env.example` en la raíz del repo). Cópialas a un archivo `.env` local (NO lo subas a git, ya está en `.gitignore`) o expórtalas en tu shell antes de abrir Jupyter.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

from src.config import LAGOS
from src import sentinel_api

### Ejercicio 1 — Conexión con la API de Sentinel-2 (openEO)

`get_openeo_connection()` intenta autenticarse por *client credentials* (silencioso, recomendado) si están definidas `SH_CLIENT_ID`/`SH_CLIENT_SECRET`; si no, cae a login interactivo por navegador (`authenticate_oidc()`), que debes completar manualmente.

In [2]:
con = sentinel_api.get_openeo_connection()
print("Conectado a:", con.root_url)
print("Usuario autenticado correctamente.")

Conectado a: https://openeo.dataspace.copernicus.eu/openeo/1.2/
Usuario autenticado correctamente.


### Ejercicio 2 — Descarga de datos raster por lago y fecha

Se descargan únicamente:
- **B03, B04, B08** (vía openEO) para calcular NDVI y NDWI localmente.
- El resultado del **script oficial de cianobacteria** de Sentinel Hub (vía Process API) para cada fecha — no se descargan bandas completas de más para este índice.

Se usan exclusivamente las fechas oficiales del enunciado (11 por lago) para minimizar tiempo de descarga y asegurar reproducibilidad entre grupos.

In [3]:
from tqdm.auto import tqdm

rutas_bandas = {}
for lago, info in LAGOS.items():
    rutas_bandas[lago] = {}
    for fecha in tqdm(info["fechas"], desc=f"Descargando bandas {lago}"):
        try:
            ruta = sentinel_api.descargar_bandas(con, lago, fecha)
            rutas_bandas[lago][fecha] = ruta
        except Exception as e:
            print(f"[{lago} - {fecha}] ERROR descargando bandas: {e}")

rutas_bandas

C:\Users\nadis\Desktop\UVG\semestre 8\Data Science\CC3084-Laboratorio-4\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando bandas atitlan:   0%|          | 0/11 [00:00<?, ?it/s]

Descargando bandas atitlan: 100%|██████████| 11/11 [00:00<00:00, 4056.74it/s]

Descargando bandas amatitlan:   0%|          | 0/11 [00:00<?, ?it/s]

Descargando bandas amatitlan: 100%|██████████| 11/11 [00:00<00:00, 1216.67it/s]

{'atitlan': {'2025-01-18': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-01-18_bandas.tif'),
  '2025-04-13': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-04-13_bandas.tif'),
  '2025-05-13': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-05-13_bandas.tif'),
  '2025-07-17': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-07-17_bandas.tif'),
  '2025-11-21': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-11-21_bandas.tif'),
  '2025-12-29': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-12-29_bandas.tif'),
  '2026-02-12': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2026-02-12_ba

In [4]:
rutas_cyano = {}
for lago, info in LAGOS.items():
    rutas_cyano[lago] = {}
    for fecha in tqdm(info["fechas"], desc=f"Descargando cianobacteria {lago}"):
        try:
            ruta = sentinel_api.descargar_cyano(lago, fecha)
            rutas_cyano[lago][fecha] = ruta
        except Exception as e:
            print(f"[{lago} - {fecha}] ERROR descargando cyano: {e}")

rutas_cyano

Descargando cianobacteria atitlan:   0%|          | 0/11 [00:00<?, ?it/s]

Descargando cianobacteria atitlan:   9%|▉         | 1/11 [00:00<00:06,  1.46it/s]

Descargando cianobacteria atitlan: 100%|██████████| 11/11 [00:00<00:00, 15.58it/s]

Descargando cianobacteria amatitlan:   0%|          | 0/11 [00:00<?, ?it/s]

Descargando cianobacteria amatitlan: 100%|██████████| 11/11 [00:00<00:00, 694.42it/s]

{'atitlan': {'2025-01-18': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-01-18_cyano.tif'),
  '2025-04-13': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-04-13_cyano.tif'),
  '2025-05-13': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-05-13_cyano.tif'),
  '2025-07-17': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-07-17_cyano.tif'),
  '2025-11-21': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-11-21_cyano.tif'),
  '2025-12-29': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2025-12-29_cyano.tif'),
  '2026-02-12': WindowsPath('C:/Users/nadis/Desktop/UVG/semestre 8/Data Science/CC3084-Laboratorio-4/data/raw/atitlan/2026-02-12_cyano.ti

### Verificación rápida

Confirmamos que los archivos se descargaron en `data/raw/<lago>/` y que tienen las bandas/dimensiones esperadas antes de pasar al Notebook 2 (índices y análisis temporal).

In [5]:
import rasterio

for lago in LAGOS:
    for fecha, ruta in rutas_bandas.get(lago, {}).items():
        with rasterio.open(ruta) as src:
            print(lago, fecha, "bandas:", src.count, "tamaño:", src.width, "x", src.height)

atitlan 2025-01-18 bandas: 3 tamaño: 2759 x 1751
atitlan 2025-04-13 bandas: 3 tamaño: 2759 x 1751
atitlan 2025-05-13 bandas: 3 tamaño: 2759 x 1751
atitlan 2025-07-17 bandas: 3 tamaño: 2759 x 1751
atitlan 2025-11-21 bandas: 3 tamaño: 2759 x 1751
atitlan 2025-12-29 bandas: 3 tamaño: 2759 x 1751
atitlan 2026-02-12 bandas: 3 tamaño: 2759 x 1751
atitlan 2026-03-24 bandas: 3 tamaño: 2759 x 1751
atitlan 2026-04-13 bandas: 3 tamaño: 2759 x 1751
atitlan 2026-04-28 bandas: 3 tamaño: 2759 x 1751
atitlan 2026-07-22 bandas: 3 tamaño: 2759 x 1751
amatitlan 2025-01-28 bandas: 3 tamaño: 1360 x 917
amatitlan 2025-04-15 bandas: 3 tamaño: 1360 x 917
amatitlan 2025-04-28 bandas: 3 tamaño: 1360 x 917
amatitlan 2025-11-24 bandas: 3 tamaño: 1360 x 917
amatitlan 2026-01-08 bandas: 3 tamaño: 1360 x 917
amatitlan 2026-02-02 bandas: 3 tamaño: 1360 x 917
amatitlan 2026-02-07 bandas: 3 tamaño: 1360 x 917
amatitlan 2026-03-29 bandas: 3 tamaño: 1360 x 917
amatitlan 2026-04-13 bandas: 3 tamaño: 1360 x 917
amatitlan 2